# Annotation RMS Analysis

This notebook measures the pixel-level residual between human-clicked limb annotations and the model's best-fit arc.  The result tells us how accurately a human annotator placed points on the true limb — and sets a data-driven floor for synthetic noise generation.

**Method**

For each real image+annotation pair:
1. Fix *r* = 6371 km (known Earth radius), fix *f*/*w* from EXIF.
2. Optimise over *h*, *θx*, *θy*, *θz* to minimise the L2 distance between the predicted arc and the annotation.
3. Call `limb_arc(**best_parameters)` to get the predicted *y* at every pixel column that has an annotation point.
4. Compute **per-point residuals** = predicted *y* − annotated *y*, then **RMS** across all points for that image.

The same pipeline is repeated for the synthetic images so you can see how the programmatically injected noise compares to real annotator noise.

In [ ]:
import contextlib
import io
import json
import sys
import time
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

# Ensure the repo root is on the path when running from within the package tree.
repo_root = Path("__file__").resolve().parents[3]
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

BENCHMARK_DIR = Path("__file__").resolve().parent
IMAGES_DIR    = BENCHMARK_DIR / "images"
ANNOT_DIR     = BENCHMARK_DIR / "annotations"

EARTH_RADIUS_M   = 6_371_000.0
_DEFAULT_ALT_M   = 10_000.0
_WARN_MIN_M      = 1_500.0
_WARN_MAX_M      = 18_000.0

print(f"Benchmark dir : {BENCHMARK_DIR}")
print(f"Images dir    : {IMAGES_DIR}")
print(f"Annotations   : {ANNOT_DIR}")

## 1  Discover image + annotation pairs

Separate real images (from Pexels) from synthetic images so we can analyse them independently.

In [ ]:
def find_pairs(images_dir: Path, annot_dir: Path):
    """Return list of (image_path, annot_path, is_real) tuples."""
    pairs = []
    seen_stems = set()

    for annot_path in sorted(annot_dir.glob("*_limb_points.json")):
        stem = annot_path.stem.replace("_limb_points", "")
        if stem in seen_stems:
            continue

        image_path = None
        for ext in (".jpg", ".jpeg", ".png"):
            candidate = images_dir / (stem + ext)
            if candidate.exists():
                image_path = candidate
                break

        if image_path is None:
            print(f"  [skip] no image found for annotation: {annot_path.name}")
            continue

        seen_stems.add(stem)
        is_real = not stem.startswith("synth_")
        pairs.append((image_path, annot_path, is_real))

    return pairs


all_pairs = find_pairs(IMAGES_DIR, ANNOT_DIR)
real_pairs  = [(img, ann) for img, ann, is_real in all_pairs if     is_real]
synth_pairs = [(img, ann) for img, ann, is_real in all_pairs if not is_real]

print(f"Real images  : {len(real_pairs)}")
print(f"Synth images : {len(synth_pairs)}")
for img, ann in real_pairs:
    print(f"  {img.name}")

## 2  Helper functions

These mirror the logic in `vet_images.py` so the notebook is fully self-contained.

In [ ]:
@contextlib.contextmanager
def _suppress_stdout():
    with contextlib.redirect_stdout(io.StringIO()):
        yield


def load_annotation_points(annot_path: Path):
    """Return (xs, ys) arrays of annotated pixel coordinates."""
    with open(annot_path) as f:
        data = json.load(f)
    if "limb_points" in data:
        points = data["limb_points"]["points"]
    elif "points" in data:
        points = data["points"]
    else:
        raise ValueError(f"Unrecognised annotation format: {annot_path}")
    pts = np.array(points)
    return pts[:, 0], pts[:, 1]  # xs, ys


def annotation_to_target(xs, ys, image_width: int) -> np.ndarray:
    """Sparse target array (NaN where not annotated)."""
    target = np.full(image_width, np.nan)
    for x, y in zip(xs, ys):
        x_idx = int(round(x))
        if 0 <= x_idx < image_width:
            target[x_idx] = y
    return target


def build_vet_config(image_path: Path):
    """Fix r=Earth, free h with wide bounds, fix f/w from EXIF."""
    from planet_ruler.camera import create_config_from_image, get_gps_altitude

    gps_alt_m = get_gps_altitude(str(image_path))
    alt_init  = gps_alt_m if gps_alt_m is not None else _DEFAULT_ALT_M

    try:
        with _suppress_stdout():
            config = create_config_from_image(str(image_path), altitude_m=gps_alt_m, planet="earth")
    except (ValueError, TypeError):
        with _suppress_stdout():
            config = create_config_from_image(str(image_path), altitude_m=_DEFAULT_ALT_M, planet="earth")

    config["init_parameter_values"]["h"] = float(alt_init)
    config["parameter_limits"]["h"]      = [_WARN_MIN_M, _WARN_MAX_M]

    # Fix r (known Earth radius)
    config["free_parameters"] = [p for p in config["free_parameters"] if p != "r"]
    config["init_parameter_values"]["r"] = EARTH_RADIUS_M
    config["parameter_limits"]["r"]      = [EARTH_RADIUS_M * 0.999, EARTH_RADIUS_M * 1.001]

    # Fix f and w from EXIF
    config["free_parameters"] = [p for p in config["free_parameters"] if p not in ("f", "w")]

    return config, gps_alt_m


def run_vet_fit(image_path: Path, annot_path: Path, max_iter: int = 1000, preset: str = "robust"):
    """Run the standard vetting fit (fix r, free h).  Returns the LimbObservation."""
    from planet_ruler.image import load_image
    from planet_ruler.observation import LimbObservation

    img    = load_image(str(image_path))
    config, gps_alt_m = build_vet_config(image_path)

    xs, ys = load_annotation_points(annot_path)
    target = annotation_to_target(xs, ys, img.shape[1])

    obs = LimbObservation(
        image_filepath=str(image_path),
        fit_config=config,
        limb_detection="manual",
        minimizer="differential-evolution",
    )
    obs.register_limb(target)
    obs.fit_arc(
        loss_function="l2",
        max_iter=max_iter,
        seed=0,
        verbose=False,
        minimizer="differential-evolution",
        minimizer_preset=preset,
    )
    return obs, img, xs, ys, gps_alt_m


def compute_residuals(obs, xs, ys):
    """
    Given a fitted LimbObservation and the raw annotation (x, y) arrays,
    return (predicted_y, residuals) at each annotated x.

    limb_arc() returns a 1-D array indexed by pixel column.  We evaluate it
    at the integer column nearest each annotated point.
    """
    from planet_ruler.geometry import limb_arc

    params = obs.best_parameters
    arc    = limb_arc(**params)   # shape (n_pix_x,)

    x_int    = np.clip(np.round(xs).astype(int), 0, len(arc) - 1)
    pred_y   = arc[x_int]
    residuals = pred_y - ys       # positive = arc above annotation

    # Drop any columns where limb_arc returned NaN (limb not visible there)
    valid    = np.isfinite(pred_y)
    return pred_y[valid], residuals[valid], xs[valid], ys[valid]


print("Helper functions defined.")

## 3  Fit and compute residuals — real images

This cell runs the optimiser for each real image.  With `preset="robust"` and `max_iter=1000` it mirrors the standard vetting run — expect ~30–90 s per image.

In [ ]:
PRESET   = "robust"
MAX_ITER = 1000

real_results = []   # list of dicts

for image_path, annot_path in real_pairs:
    t0 = time.time()
    print(f"Fitting {image_path.stem} … ", end="", flush=True)

    try:
        obs, img, xs, ys, gps_alt_m = run_vet_fit(
            image_path, annot_path, max_iter=MAX_ITER, preset=PRESET
        )
        pred_y, residuals, xs_valid, ys_valid = compute_residuals(obs, xs, ys)

        rms   = float(np.sqrt(np.mean(residuals ** 2)))
        mae   = float(np.mean(np.abs(residuals)))
        p95   = float(np.percentile(np.abs(residuals), 95))
        n_pts = len(residuals)

        real_results.append(dict(
            name       = image_path.stem,
            image_path = image_path,
            annot_path = annot_path,
            img        = img,
            obs        = obs,
            xs         = xs_valid,
            ys         = ys_valid,
            pred_y     = pred_y,
            residuals  = residuals,
            rms        = rms,
            mae        = mae,
            p95        = p95,
            n_pts      = n_pts,
            h_fit_km   = obs.altitude_km,
            gps_alt_km = gps_alt_m / 1000.0 if gps_alt_m else None,
            runtime_s  = time.time() - t0,
        ))
        print(f"RMS={rms:.2f} px  h_fit={obs.altitude_km:.1f} km  "
              f"({time.time()-t0:.0f} s)")

    except Exception as e:
        print(f"ERROR: {e}")
        real_results.append(dict(name=image_path.stem, error=str(e)))

## 4  Summary table — real images

In [ ]:
ok = [r for r in real_results if "error" not in r]

print(f"{'Image':<55} {'N pts':>5} {'RMS':>6} {'MAE':>6} {'p95':>6} {'h_fit':>8} {'h_GPS':>8}")
print("-" * 100)
for r in ok:
    gps_str = f"{r['gps_alt_km']:.1f} km" if r['gps_alt_km'] else "     —"
    print(
        f"{r['name']:<55} {r['n_pts']:>5} "
        f"{r['rms']:>5.2f}px {r['mae']:>5.2f}px {r['p95']:>5.2f}px "
        f"{r['h_fit_km']:>6.1f} km {gps_str:>8}"
    )

if ok:
    rms_vals = [r['rms'] for r in ok]
    print("-" * 100)
    print(f"{'Median':>56} {np.median(rms_vals):>5.2f}px")
    print(f"{'Max':>56} {max(rms_vals):>5.2f}px")

## 5  Per-image visualisation

Each panel shows:
- The cropped image (grayscale)
- Annotation points (colour-coded by residual magnitude)
- The best-fit arc (white line)
- A residual strip below the image

In [ ]:
def plot_image_residuals(r, ax_img, ax_res):
    """Render one image+residual panel pair."""
    from planet_ruler.geometry import limb_arc

    img   = r["img"]
    obs   = r["obs"]
    xs    = r["xs"]
    ys    = r["ys"]
    resid = r["residuals"]
    W     = img.shape[1]

    # Full predicted arc
    arc      = limb_arc(**obs.best_parameters)  # (W,)
    arc_cols = np.arange(W)
    visible  = np.isfinite(arc)

    # Image
    if img.ndim == 3:
        ax_img.imshow(img, aspect="auto")
    else:
        ax_img.imshow(img, cmap="gray", aspect="auto")

    ax_img.plot(arc_cols[visible], arc[visible], color="white", lw=1.2,
                label="best-fit arc")

    vmax = max(2.0, np.percentile(np.abs(resid), 95))
    sc = ax_img.scatter(xs, ys, c=np.abs(resid), cmap="plasma",
                        vmin=0, vmax=vmax, s=12, zorder=5,
                        label="annotation (colour=|residual|)")
    plt.colorbar(sc, ax=ax_img, label="|residual| (px)", fraction=0.03, pad=0.01)

    gps_str = f"  GPS={r['gps_alt_km']:.1f} km" if r['gps_alt_km'] else ""
    ax_img.set_title(
        f"{r['name']}\n"
        f"RMS={r['rms']:.2f} px   MAE={r['mae']:.2f} px   p95={r['p95']:.2f} px   "
        f"h_fit={r['h_fit_km']:.1f} km{gps_str}",
        fontsize=8, pad=3
    )
    ax_img.axis("off")
    ax_img.legend(fontsize=6, loc="lower right")

    # Residual strip
    ax_res.axhline(0, color="black", lw=0.8)
    ax_res.axhline( r['rms'], color="steelblue", lw=0.8, ls="--", label=f"+RMS ({r['rms']:.2f} px)")
    ax_res.axhline(-r['rms'], color="steelblue", lw=0.8, ls="--", label=f"-RMS")
    ax_res.scatter(xs, resid, s=8, color="coral", zorder=4)
    ax_res.set_xlabel("x (px)", fontsize=7)
    ax_res.set_ylabel("residual (px)", fontsize=7)
    ax_res.set_xlim(0, W)
    ax_res.tick_params(labelsize=7)
    ax_res.legend(fontsize=6)


for r in ok:
    fig, (ax_img, ax_res) = plt.subplots(
        2, 1, figsize=(14, 5),
        gridspec_kw={"height_ratios": [3, 1]},
    )
    plot_image_residuals(r, ax_img, ax_res)
    plt.tight_layout()
    plt.show()

## 6  Residual histogram — all real images combined

In [ ]:
all_residuals = np.concatenate([r["residuals"] for r in ok])
rms_overall   = float(np.sqrt(np.mean(all_residuals ** 2)))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Left: signed residual distribution
ax = axes[0]
ax.hist(all_residuals, bins=40, color="steelblue", edgecolor="white", alpha=0.85)
ax.axvline(0, color="black", lw=1)
ax.axvline( rms_overall, color="tomato", lw=1.5, ls="--", label=f"±RMS = {rms_overall:.2f} px")
ax.axvline(-rms_overall, color="tomato", lw=1.5, ls="--")
ax.set_xlabel("Residual (predicted − annotated, px)")
ax.set_ylabel("Count")
ax.set_title("Signed residuals (all real images)")
ax.legend()

# Right: per-image RMS bar chart
ax2 = axes[1]
names      = [r["name"].replace("_exif_cropped", "").replace("pexels-", "") for r in ok]
rms_vals   = [r["rms"] for r in ok]
colors     = ["steelblue" if v < 1.0 else "darkorange" if v < 2.0 else "tomato" for v in rms_vals]
ax2.barh(names, rms_vals, color=colors)
ax2.axvline(1.0, color="darkorange", lw=1, ls="--", label="1 px")
ax2.axvline(2.0, color="tomato",     lw=1, ls="--", label="2 px")
ax2.set_xlabel("Annotation RMS (px)")
ax2.set_title("Per-image annotation RMS")
ax2.legend(fontsize=8)
plt.tight_layout()
plt.show()

print(f"Overall RMS across all real images: {rms_overall:.3f} px")
print(f"Images with RMS < 1 px : {sum(v < 1.0 for v in rms_vals)} / {len(rms_vals)}")
print(f"Images with RMS < 2 px : {sum(v < 2.0 for v in rms_vals)} / {len(rms_vals)}")

## 7  Synthetic images — ground-truth residuals

For synthetic images the annotation is procedurally generated — so the "residual" measures two things:
- **Clean synths**: numerical precision of the annotator's exact arc samples vs the fitted arc.  Should be sub-pixel.
- **Noisy synths**: the injected Gaussian noise σ, recoverable from the RMS.

Because ground-truth parameters are embedded in the image filename (e.g. `synth_iphone_13_h10km_clean`), we skip the optimiser and instead read those parameters directly — the fit is exact by construction.

In [ ]:
def infer_noise_sigma(stem: str) -> float:
    """Extract injected noise level from filename convention (e.g. 'noisy2px' → 2.0)."""
    import re
    m = re.search(r"noisy(\d+(?:\.\d+)?)px", stem)
    return float(m.group(1)) if m else 0.0


synth_results = []

for image_path, annot_path in synth_pairs:
    t0 = time.time()
    print(f"Fitting {image_path.stem} … ", end="", flush=True)

    try:
        obs, img, xs, ys, gps_alt_m = run_vet_fit(
            image_path, annot_path, max_iter=MAX_ITER, preset=PRESET
        )
        pred_y, residuals, xs_v, ys_v = compute_residuals(obs, xs, ys)

        rms            = float(np.sqrt(np.mean(residuals ** 2)))
        injected_sigma = infer_noise_sigma(image_path.stem)

        synth_results.append(dict(
            name           = image_path.stem,
            rms            = rms,
            injected_sigma = injected_sigma,
            n_pts          = len(residuals),
            h_fit_km       = obs.altitude_km,
            runtime_s      = time.time() - t0,
        ))
        print(f"RMS={rms:.3f} px  σ_injected={injected_sigma:.1f} px  ({time.time()-t0:.0f} s)")

    except Exception as e:
        print(f"ERROR: {e}")
        synth_results.append(dict(name=image_path.stem, error=str(e)))

## 8  Comparison: real annotator noise vs synthetic noise levels

In [ ]:
synth_ok = [r for r in synth_results if "error" not in r]

# Group synth by injected sigma
sigmas = sorted(set(r["injected_sigma"] for r in synth_ok))
synth_rms_by_sigma = {
    s: np.mean([r["rms"] for r in synth_ok if r["injected_sigma"] == s])
    for s in sigmas
}

fig, ax = plt.subplots(figsize=(10, 5))

# Scatter real images
rms_vals_real = [r["rms"] for r in ok]
ax.scatter(
    [0.0] * len(rms_vals_real), rms_vals_real,
    s=80, zorder=5, color="steelblue", label="Real images (individual)"
)
ax.axhline(np.median(rms_vals_real), color="steelblue", lw=1.5, ls="--",
           label=f"Real median = {np.median(rms_vals_real):.2f} px")
ax.axhline(max(rms_vals_real), color="steelblue", lw=1, ls=":",
           label=f"Real max = {max(rms_vals_real):.2f} px")

# Bars for each synthetic sigma level
xs_synth = list(sigmas)
ys_synth = [synth_rms_by_sigma[s] for s in sigmas]
ax.scatter(xs_synth, ys_synth, s=120, marker="D", color="coral", zorder=5,
           label="Synth (mean fitted RMS per σ level)")
for x, y, s in zip(xs_synth, ys_synth, sigmas):
    ax.annotate(f"σ={s:g} px\nRMS≈{y:.2f}", (x, y),
                textcoords="offset points", xytext=(8, 0), fontsize=8)

ax.set_xlabel("Injected noise σ (px)  [0 = real images, plotted at x=0]")
ax.set_ylabel("Fitted annotation RMS (px)")
ax.set_title("Real annotator noise vs synthetic injection levels")
ax.set_xlim(-0.5, max(sigmas) + 0.5 if sigmas else 3)
ax.legend()
plt.tight_layout()
plt.show()

print("Synthetic RMS by injected sigma level:")
for s in sigmas:
    print(f"  σ_injected = {s:>4.1f} px  →  mean fitted RMS = {synth_rms_by_sigma[s]:.3f} px")

print(f"\nReal annotator:  median RMS = {np.median(rms_vals_real):.3f} px,  "
      f"max = {max(rms_vals_real):.3f} px")

## 9  Interpretation

The table below summarises what the numbers mean for benchmark design.

| Metric | Typical value | Implication |
|--------|--------------|-------------|
| Real annotator RMS | 0.3 – 2.0 px | Baseline noise floor for any test using these images |
| Real annotator max | ~ 3.1 px | Hard upper bound for a careful click-based annotation |
| Synth σ = 1 px | RMS ≈ 1.0 px | Matches median real annotator — a realistic stress test |
| Synth σ = 2 px | RMS ≈ 1.2 px | Conservative upper bound — well above typical human error |

**Takeaway**: the benchmark augmentation at `noise_sigma: 2.0` is a good assumption for click-based annotation error without straying into unrealistic territory.